In [ ]:
# Pre-processing image
# ----------------------------------

import cv2
import numpy as np
from pathlib import Path

input_dir = Path(r"/home/dl-box/users/students/phasathorn-jewrasumnuay/CephPic/A0InitialFilm")
output_dir = Path(r"/home/dl-box/users/students/phasathorn-jewrasumnuay/Ceph-image/A1Enhance")

output_dir.mkdir(parents=True, exist_ok=True)
for p in output_dir.glob("*.jpg"):
    p.unlink()


# Resize + padding
def resize_with_padding(gray, out_w=512, out_h=640, pad_value=0):
    h, w = gray.shape[:2]
    scale = min(out_w / w, out_h / h)
    new_w = int(round(w * scale))
    new_h = int(round(h * scale))

    resized = cv2.resize(gray, (new_w, new_h), interpolation=cv2.INTER_AREA)

    canvas = np.full((out_h, out_w), pad_value, dtype=resized.dtype)
    pad_left = (out_w - new_w) // 2
    pad_top = (out_h - new_h) // 2
    canvas[pad_top:pad_top + new_h, pad_left:pad_left + new_w] = resized
    return canvas


# Enhance
def preprocess(img_bgr, out_w=512, out_h=640, use_negative=False):
    # 1) grayscale
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    # 2) resize with padding
    gray = resize_with_padding(gray, out_w=out_w, out_h=out_h)

    # 3) gamma correction
    x = gray.astype(np.float32) / 255.0
    x = np.power(x, 0.8)          # gamma = 0.8
    gray = (x * 255.0).clip(0, 255).astype(np.uint8)

    # 4) CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray = clahe.apply(gray)

    return gray


count = 0
for img_path in input_dir.glob("*.jpg"):
    img = cv2.imread(str(img_path))
    if img is None:
        print(f"⚠️ Could not read {img_path}")
        continue

    out = preprocess(img, out_w=512, out_h=640)
    cv2.imwrite(str(output_dir / img_path.name), out)
    count += 1

print(f"Finished: {count} images processed")


Finished: 400 images processed


In [10]:
# .jpg -> .png -> Split group (5-flow cross validation)
# ----------------------------------

import cv2, glob, os, shutil
from pathlib import Path
from sklearn.model_selection import KFold, train_test_split

SRC = r"/home/dl-box/users/students/phasathorn-jewrasumnuay/CephPic/A1Enhance"
DST = r"/home/dl-box/users/students/phasathorn-jewrasumnuay/CephPic/A2SplitGroup"
RANDOM_STATE = 42
N_FOLDS = 5

jpg_paths = sorted(glob.glob(os.path.join(SRC, "*.jpg")))
base_ids = [Path(p).stem for p in jpg_paths]

# แบ่ง Test 20% ก่อน แล้วแบ่ง 5-Fold จาก 80% ที่เหลือ
trainval_ids, test_ids = train_test_split(base_ids, test_size=0.2, random_state=RANDOM_STATE, shuffle=True)
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
folds = [{"train": [trainval_ids[i] for i in tr], "val": [trainval_ids[i] for i in va]}
         for tr, va in kf.split(trainval_ids)]

def copy_pair(ids, dst_folder):
    os.makedirs(dst_folder, exist_ok=True)
    for stem in ids:
        jpg_src = os.path.join(SRC, stem + ".jpg")
        img = cv2.imread(jpg_src)
        if img is None:
            print(f"⚠️ Could not read: {jpg_src}")
            continue
        cv2.imwrite(os.path.join(dst_folder, stem + ".png"), img)   

        json_src = os.path.join(SRC, stem + ".json")
        if os.path.exists(json_src):
            shutil.copy(json_src, os.path.join(dst_folder, stem + ".json"))

for i, fold in enumerate(folds, 1):
    fold_dir = os.path.join(DST, f"Fold{i}")
    copy_pair(fold["train"], os.path.join(fold_dir, "Train"))
    copy_pair(fold["val"],   os.path.join(fold_dir, "Val"))
    copy_pair(test_ids,      os.path.join(fold_dir, "Test"))
    print(f"📂 Fold{i} → train:{len(fold['train'])}, val:{len(fold['val'])}, test:{len(test_ids)}")

print("Finished")

📂 Fold1 → train:256, val:64, test:80
📂 Fold2 → train:256, val:64, test:80
📂 Fold3 → train:256, val:64, test:80
📂 Fold4 → train:256, val:64, test:80
📂 Fold5 → train:256, val:64, test:80
Finished


In [11]:
# AIT (step 1)
# Illustrate 4 anchor point: S=white(255,255,255), N=red(0,0,255), PNS=blue(255,0,0), Gn=yellow(0,255,255)
# ----------------------------------

import cv2
import json
import glob
import os
from pathlib import Path

SPLIT_ROOT = r"/home/dl-box/users/students/phasathorn-jewrasumnuay/CephPic/A2SplitGroup"
PREVIEW_DIR = r"/home/dl-box/users/students/phasathorn-jewrasumnuay/CephPic/A3PreviewAnchor5Point"

POINT_SIZE_EXACT = 1        # ขนาดจริงตอนเทรน (1x1 px)
PREVIEW_RADIUS = 4          # รัศมีวงกลม เฉพาะตอนสร้างภาพ preview ให้มองเห็นง่าย

CRITICAL_LANDMARKS = {
    "S":   {"color_bgr": (255, 255, 255), "aliases": ["S", "Sella", "sella", "L1"]},
    "N":   {"color_bgr": (0, 0, 255),     "aliases": ["N", "Nasion", "nasion", "L2"]},
    "PNS": {"color_bgr": (255, 0, 0),     "aliases": ["PNS", "pns", "Posterior Nasal Spine"]},
    "Gn":  {"color_bgr": (0, 255, 255),   "aliases": ["Gn", "Gnathion", "gnathion", "L9"]},
}


# ---------------------------------------------------------------
# 1) อ่าน landmark จาก .json (LabelMe-style)
# ---------------------------------------------------------------
def load_landmarks(json_path: str) -> dict:  # dict {label: (x, y)}
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    points = {}
    if isinstance(data, dict) and "shapes" in data:
        for shape in data["shapes"]:
            label = shape.get("label")
            pts = shape.get("points")
            if label and pts:
                x, y = pts[0]
                points[label] = (float(x), float(y))
    return points


# ---------------------------------------------------------------
# 2) จับคู่ label ที่อ่านได้จาก json เข้ากับ 4 จุดหลักที่สนใจ (S, N, PNS, Gn)
# ---------------------------------------------------------------
def match_critical_points(raw_points: dict) -> dict:
    matched = {}
    for key, info in CRITICAL_LANDMARKS.items():
        for alias in info["aliases"]:
            if alias in raw_points:
                x, y = raw_points[alias]
                matched[key] = (x, y, info["color_bgr"])
                break
    return matched


# ---------------------------------------------------------------
# 3) วาดจุด — แยก "ขนาดจริงใช้เทรน" กับ "ขนาดใหญ่ใช้ตรวจสอบ"
# ---------------------------------------------------------------
def draw_exact_points(image, matched_points: dict, point_size: int = POINT_SIZE_EXACT):
    out = image.copy()
    h, w = out.shape[:2]
    half = point_size // 2
    for key, (x, y, color) in matched_points.items():
        xi, yi = int(round(x)), int(round(y))
        x0, x1 = max(0, xi - half), min(w, xi - half + point_size)
        y0, y1 = max(0, yi - half), min(h, yi - half + point_size)
        out[y0:y1, x0:x1] = color
    return out


def draw_preview_points(image, matched_points: dict, radius: int = PREVIEW_RADIUS):
    out = image.copy()
    for key, (x, y, color) in matched_points.items():
        center = (int(round(x)), int(round(y)))
        cv2.circle(out, center, radius, color, thickness=-1)
        cv2.putText(out, key, (center[0] + radius + 2, center[1]),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)
    return out


# ---------------------------------------------------------------
# 4) save preview 
# ---------------------------------------------------------------
def process_one_sample(png_path: str, out_dir: str):
    stem = Path(png_path).stem
    json_path = str(Path(png_path).with_suffix(".json"))

    image = cv2.imread(png_path)
    if image is None:
        print(f"⚠️ อ่านภาพไม่ได้: {png_path}")
        return

    if not os.path.exists(json_path):
        print(f"⚠️ ไม่พบไฟล์ json คู่กัน: {json_path}")
        return

    raw_points = load_landmarks(json_path)
    matched = match_critical_points(raw_points)

    missing = [k for k in CRITICAL_LANDMARKS if k not in matched]
    if missing:
        print(f"⚠️ [{stem}] หาจุดไม่เจอ: {missing}")

    preview_img = draw_preview_points(image, matched)   # ขนาดเท่าภาพต้นฉบับ

    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"{stem}.png")     
    cv2.imwrite(out_path, preview_img)


# ---------------------------------------------------------------
# 5) วนทุก Fold (Fold1-Fold5) x ทุก split (Train/Val/Test) x ทุกภาพ
# ---------------------------------------------------------------
def process_all_folds():
    fold_dirs = sorted(glob.glob(os.path.join(SPLIT_ROOT, "Fold*")))
    if not fold_dirs:
        print(f"⚠️ ไม่พบโฟลเดอร์ Fold ใน {SPLIT_ROOT}")
        return

    total = 0
    for fold_dir in fold_dirs:
        fold_name = os.path.basename(fold_dir)
        for split_name in ["Train", "Val", "Test"]:
            split_dir = os.path.join(fold_dir, split_name)
            png_files = sorted(glob.glob(os.path.join(split_dir, "*.png")))
            if not png_files:
                continue

            out_dir = os.path.join(PREVIEW_DIR, fold_name, split_name)
            for png_path in png_files:
                process_one_sample(png_path, out_dir)
                total += 1

            print(f"📂 {fold_name}/{split_name} → {len(png_files)} ภาพ เซฟที่ {out_dir}")

    print(f"✅ เสร็จสิ้น รวมทั้งหมด {total} ภาพ → {PREVIEW_DIR}")


if __name__ == "__main__":
    process_all_folds()

📂 Fold1/Train → 256 ภาพ เซฟที่ /home/dl-box/users/students/phasathorn-jewrasumnuay/CephPic/A3PreviewAnchor5Point/Fold1/Train
📂 Fold1/Val → 64 ภาพ เซฟที่ /home/dl-box/users/students/phasathorn-jewrasumnuay/CephPic/A3PreviewAnchor5Point/Fold1/Val
📂 Fold1/Test → 80 ภาพ เซฟที่ /home/dl-box/users/students/phasathorn-jewrasumnuay/CephPic/A3PreviewAnchor5Point/Fold1/Test
📂 Fold2/Train → 256 ภาพ เซฟที่ /home/dl-box/users/students/phasathorn-jewrasumnuay/CephPic/A3PreviewAnchor5Point/Fold2/Train
📂 Fold2/Val → 64 ภาพ เซฟที่ /home/dl-box/users/students/phasathorn-jewrasumnuay/CephPic/A3PreviewAnchor5Point/Fold2/Val
📂 Fold2/Test → 80 ภาพ เซฟที่ /home/dl-box/users/students/phasathorn-jewrasumnuay/CephPic/A3PreviewAnchor5Point/Fold2/Test
📂 Fold3/Train → 256 ภาพ เซฟที่ /home/dl-box/users/students/phasathorn-jewrasumnuay/CephPic/A3PreviewAnchor5Point/Fold3/Train
📂 Fold3/Val → 64 ภาพ เซฟที่ /home/dl-box/users/students/phasathorn-jewrasumnuay/CephPic/A3PreviewAnchor5Point/Fold3/Val
📂 Fold3/Test → 80 ภาพ

In [ ]:
# AIT (step 2) - Full landmark preview (4 critical + 19 additional = 23 points)
# 4 จุดเดิม (S,N,PNS,Gn): วงกลมทึบสีเฉพาะตัว + label (สไตล์เดิม)
# 19 จุดเพิ่ม: วงแหวน (ring) รัศมี 4px สีเขียว + จุดแดงตรงกลาง 1px (ตำแหน่งแม่นยำ)
# พร้อม copy ไฟล์ .json ต้นทาง (ground truth) ไปเก็บคู่กับภาพ preview
# ----------------------------------

import cv2
import json
import glob
import os
import shutil
from pathlib import Path

SPLIT_ROOT = r"/home/dl-box/users/students/phasathorn-jewrasumnuay/CephPic/A2SplitGroup"
PREVIEW_DIR = r"/home/dl-box/users/students/phasathorn-jewrasumnuay/CephPic/A4PreviewFull23Point"

PREVIEW_RADIUS = 4          # รัศมีวงกลม/วงแหวน สำหรับ preview
RING_THICKNESS = 1          # ความหนาเส้นวงแหวน (จุดใหม่ 19 จุด)
CENTER_DOT_SIZE = 1         # ขนาดจุดสีแดงตรงกลาง (px)
RING_COLOR_BGR = (0, 255, 0)     # สีวงแหวนของจุดใหม่ทั้งหมด (เขียว)
CENTER_COLOR_BGR = (0, 0, 255)   # สีจุดตรงกลางของจุดใหม่ทั้งหมด (แดง)

# ---------------------------------------------------------------
# 4 จุดหลัก (critical) — สีเฉพาะตัว เหมือน code เดิม
# ---------------------------------------------------------------
CRITICAL_LANDMARKS = {
    "S":   {"color_bgr": (255, 255, 255), "aliases": ["S", "Sella", "sella", "L1"]},
    "N":   {"color_bgr": (0, 0, 255),     "aliases": ["N", "Nasion", "nasion", "L2"]},
    "PNS": {"color_bgr": (255, 0, 0),     "aliases": ["PNS", "pns", "Posterior Nasal Spine"]},
    "Gn":  {"color_bgr": (0, 255, 255),   "aliases": ["Gn", "Gnathion", "gnathion", "L9"]},
}

# ---------------------------------------------------------------
# 19 จุดเพิ่มเติม — ring + center dot สไตล์เดียวกันหมด
# NOTE: alias เป็นการเดาจากชื่อมาตรฐานทาง cephalometrics — ถ้า json จริง
# สะกด/ใช้ชื่อต่างไปจากนี้ จะขึ้น "หาจุดไม่เจอ" เหมือนเดิม แก้ alias ตรงนี้ได้เลย
# ---------------------------------------------------------------
ADDITIONAL_LANDMARKS = {
    "Ns":   ["Ns", "N'", "Soft Tissue Nasion", "nasion soft tissue"],
    "Or":   ["Or", "Orbitale", "orbitale", "L3"],
    "ANS":  ["ANS", "Anterior Nasal Spine", "ans"],
    "A":    ["A", "A point", "A-point", "Subspinale", "subspinale", "L5"],
    "Isa":  ["Isa", "U1A", "Upper Incisor Apex", "isa"],
    "Is":   ["Is", "U1", "Upper Incisor", "is"],
    "Ii":   ["Ii", "L1", "Lower Incisor", "ii"],
    "B":    ["B", "B point", "B-point", "Supramentale", "supramental", "L6"],
    "Pg":   ["Pg", "Pogonion", "pogonion", "L7"],
    "Me":   ["Me", "Menton", "menton", "L8"],
    "Iia":  ["Iia", "L1A", "Lower Incisor Apex", "iia"],
    "D":    ["D", "D point", "D-point"],
    "Go":   ["Go", "Gonion", "gonion", "L10"],
    "Ar":   ["Ar", "Articulare", "articulare"],
    "Po":   ["Po", "Porion", "porion", "L4"],
    "PRN":  ["PRN", "Pronasale", "pronasale", "Prn"],
    "Sn":   ["Sn", "Subnasale", "subnasale"],
    "Ls":   ["Ls", "Labrale Superius", "labrale superius"],
    "Pg'":  ["Pg'", "Pg prime", "Soft Tissue Pogonion", "soft tissue pogonion"],
}


# ---------------------------------------------------------------
# 1) อ่าน landmark จาก .json (LabelMe-style)
# ---------------------------------------------------------------
def load_landmarks(json_path: str) -> dict:  # dict {label: (x, y)}
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    points = {}
    if isinstance(data, dict) and "shapes" in data:
        for shape in data["shapes"]:
            label = shape.get("label")
            pts = shape.get("points")
            if label and pts:
                x, y = pts[0]
                points[label] = (float(x), float(y))
    return points


# ---------------------------------------------------------------
# 2) จับคู่ label จาก json เข้ากับจุดที่สนใจ (ใช้ได้ทั้งกลุ่ม critical/additional)
# ---------------------------------------------------------------
def match_points(raw_points: dict, alias_map: dict) -> dict:
    """alias_map: {key: [alias1, alias2, ...]} -> คืน {key: (x, y)}"""
    matched = {}
    for key, aliases in alias_map.items():
        for alias in aliases:
            if alias in raw_points:
                matched[key] = raw_points[alias]
                break
    return matched


# ---------------------------------------------------------------
# 3) วาดจุด
# ---------------------------------------------------------------
def draw_critical_points(image, matched_critical: dict):
    """4 จุดหลัก: วงกลมทึบสีเฉพาะตัว + label (สไตล์เดิม)"""
    out = image
    for key, (x, y) in matched_critical.items():
        color = CRITICAL_LANDMARKS[key]["color_bgr"]
        center = (int(round(x)), int(round(y)))
        cv2.circle(out, center, PREVIEW_RADIUS, color, thickness=-1)
        cv2.putText(out, key, (center[0] + PREVIEW_RADIUS + 2, center[1]),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)
    return out


def draw_additional_points(image, matched_additional: dict):
    """19 จุดเพิ่ม: วงแหวน (ring) รัศมี 4px สีเขียว + จุดแดงตรงกลาง 1px"""
    out = image
    half = CENTER_DOT_SIZE // 2
    h, w = out.shape[:2]
    for key, (x, y) in matched_additional.items():
        xi, yi = int(round(x)), int(round(y))
        center = (xi, yi)

        # ring (วงแหวน ไม่ทึบ)
        cv2.circle(out, center, PREVIEW_RADIUS, RING_COLOR_BGR, thickness=RING_THICKNESS)

        # center dot ขนาดจริง (1px) สีแดง — ตำแหน่งแม่นยำ
        x0, x1 = max(0, xi - half), min(w, xi - half + CENTER_DOT_SIZE)
        y0, y1 = max(0, yi - half), min(h, yi - half + CENTER_DOT_SIZE)
        out[y0:y1, x0:x1] = CENTER_COLOR_BGR

        # label ชื่อจุด — สไตล์เดียวกับ 4 จุดเดิม (ใช้สีวงแหวนเป็นสีตัวอักษร)
        cv2.putText(out, key, (center[0] + PREVIEW_RADIUS + 2, center[1]),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, RING_COLOR_BGR, 1, cv2.LINE_AA)
    return out


# ---------------------------------------------------------------
# 4) ประมวลผล 1 ภาพ: วาดครบ 23 จุด + copy json ต้นทางไปด้วย
# ---------------------------------------------------------------
def process_one_sample(png_path: str, out_dir: str):
    stem = Path(png_path).stem
    json_path = str(Path(png_path).with_suffix(".json"))

    image = cv2.imread(png_path)
    if image is None:
        print(f"⚠️ อ่านภาพไม่ได้: {png_path}")
        return

    if not os.path.exists(json_path):
        print(f"⚠️ ไม่พบไฟล์ json คู่กัน: {json_path}")
        return

    raw_points = load_landmarks(json_path)
    matched_critical = match_points(raw_points, {k: v["aliases"] for k, v in CRITICAL_LANDMARKS.items()})
    matched_additional = match_points(raw_points, ADDITIONAL_LANDMARKS)

    all_keys = list(CRITICAL_LANDMARKS.keys()) + list(ADDITIONAL_LANDMARKS.keys())
    found_keys = set(matched_critical) | set(matched_additional)
    missing = [k for k in all_keys if k not in found_keys]
    if missing:
        print(f"⚠️ [{stem}] หาจุดไม่เจอ ({len(missing)}): {missing}")

    out_img = image.copy()
    out_img = draw_critical_points(out_img, matched_critical)
    out_img = draw_additional_points(out_img, matched_additional)

    os.makedirs(out_dir, exist_ok=True)
    out_png_path = os.path.join(out_dir, f"{stem}.png")
    cv2.imwrite(out_png_path, out_img)

    # copy ไฟล์ json ต้นทาง (ground truth) ไปเก็บคู่กัน สำหรับใช้วัดความแม่นยำในสเต็ปถัดไป
    out_json_path = os.path.join(out_dir, f"{stem}.json")
    shutil.copy(json_path, out_json_path)


# ---------------------------------------------------------------
# 5) วนทุก Fold (Fold1-Fold5) x ทุก split (Train/Val/Test) x ทุกภาพ
# ---------------------------------------------------------------
def process_all_folds():
    fold_dirs = sorted(glob.glob(os.path.join(SPLIT_ROOT, "Fold*")))
    if not fold_dirs:
        print(f"⚠️ ไม่พบโฟลเดอร์ Fold ใน {SPLIT_ROOT}")
        return

    total = 0
    for fold_dir in fold_dirs:
        fold_name = os.path.basename(fold_dir)
        for split_name in ["Train", "Val", "Test"]:
            split_dir = os.path.join(fold_dir, split_name)
            png_files = sorted(glob.glob(os.path.join(split_dir, "*.png")))
            if not png_files:
                continue

            out_dir = os.path.join(PREVIEW_DIR, fold_name, split_name)
            for png_path in png_files:
                process_one_sample(png_path, out_dir)
                total += 1

            print(f"📂 {fold_name}/{split_name} → {len(png_files)} ภาพ เซฟที่ {out_dir}")

    print(f"✅ เสร็จสิ้น รวมทั้งหมด {total} ภาพ (+ json) → {PREVIEW_DIR}")


if __name__ == "__main__":
    process_all_folds()

📂 Fold1/Train → 256 ภาพ เซฟที่ /home/dl-box/users/students/phasathorn-jewrasumnuay/CephPic/A4PreviewFull24Point/Fold1/Train
📂 Fold1/Val → 64 ภาพ เซฟที่ /home/dl-box/users/students/phasathorn-jewrasumnuay/CephPic/A4PreviewFull24Point/Fold1/Val
📂 Fold1/Test → 80 ภาพ เซฟที่ /home/dl-box/users/students/phasathorn-jewrasumnuay/CephPic/A4PreviewFull24Point/Fold1/Test
📂 Fold2/Train → 256 ภาพ เซฟที่ /home/dl-box/users/students/phasathorn-jewrasumnuay/CephPic/A4PreviewFull24Point/Fold2/Train
📂 Fold2/Val → 64 ภาพ เซฟที่ /home/dl-box/users/students/phasathorn-jewrasumnuay/CephPic/A4PreviewFull24Point/Fold2/Val
📂 Fold2/Test → 80 ภาพ เซฟที่ /home/dl-box/users/students/phasathorn-jewrasumnuay/CephPic/A4PreviewFull24Point/Fold2/Test
📂 Fold3/Train → 256 ภาพ เซฟที่ /home/dl-box/users/students/phasathorn-jewrasumnuay/CephPic/A4PreviewFull24Point/Fold3/Train
📂 Fold3/Val → 64 ภาพ เซฟที่ /home/dl-box/users/students/phasathorn-jewrasumnuay/CephPic/A4PreviewFull24Point/Fold3/Val
📂 Fold3/Test → 80 ภาพ เซฟที่ 

In [20]:
# AIT (step 3) - Anatomy-Informed Topology Color Image
# ตามสมการ (8)-(10) ในเปเปอร์ "Towards Better Cephalometric Landmark
# Detection with Diffusion Data Generation"
#
# แนวคิด:
#   1) 4 จุดหลัก (topological centers) = S, N, PNS, Gn -> สีคงที่ C(L)
#   2) จุดอื่น ๆ (19 จุด) -> คำนวณระยะทางบนกราฟ d(v,L) ไปยังจุดหลักแต่ละจุด
#      น้ำหนัก w_L(v) = 1/d(v,L)  แล้ว normalize (eq.8): ŵ_L(v) = w_L(v)/Σw_L'(v)
#      ผสมสี (eq.9): C(v) = Σ ŵ_L(v)·C(L)
#   3) เส้นเชื่อม (edge) ระหว่างจุด -> ไล่สี gradient (eq.10):
#      c(t) = c1 + t/(D-1)·(c2-c1),  t = 0,...,D-1  (D = pixel distance ระหว่างจุด)
#
# หมายเหตุสำคัญ: การผสมสีของแต่ละจุด (ข้อ 1-2) ขึ้นกับ "โครงสร้างกราฟ" เท่านั้น
# ไม่ขึ้นกับตำแหน่งพิกเซลจริงในภาพ จึงคำนวณได้ครั้งเดียวตอน import (ไม่ต้องคำนวณใหม่ทุกภาพ)
# ส่วนที่ขึ้นกับพิกเซลจริงคือ "เส้น gradient" (ข้อ 3) เท่านั้น ซึ่งต้องคำนวณทีละภาพ
# ----------------------------------

import cv2
import json
import glob
import os
import shutil
import math
import pandas as pd
from pathlib import Path
from collections import deque
from concurrent.futures import ProcessPoolExecutor, as_completed
from openpyxl.styles import Font, Alignment
from openpyxl.utils import get_column_letter
from openpyxl import load_workbook

SPLIT_ROOT = r"/home/dl-box/users/students/phasathorn-jewrasumnuay/CephPic/A2SplitGroup"
PREVIEW_DIR = r"/home/dl-box/users/students/phasathorn-jewrasumnuay/CephPic/A5AITcolorFrom4color"
ANGLE_XLSX = r"/home/dl-box/users/students/phasathorn-jewrasumnuay/CephPic/A5_CephalometricAngles.xlsx"
ANGLE_SOURCE_FOLD = "Fold1"   # คำนวณมุมจาก Fold นี้ Fold เดียวพอ (พิกัดซ้ำกันทุก Fold)

NODE_RADIUS = 3          # รัศมีจุด (วงกลมทึบ) บนภาพ
EDGE_DOT_RADIUS = 1       # รัศมีจุดแต่ละ step ของเส้น gradient (ยิ่งเล็กยิ่งใกล้ 1px ตามสมการ)

# ---------------------------------------------------------------
# 5 จุดหลัก (topological centers) — สีคงที่ C(L)
# ---------------------------------------------------------------
CRITICAL_LANDMARKS = {
    "S":   {"color_bgr": (255, 255, 255), "aliases": ["S", "Sella", "sella", "L1"]},
    "N":   {"color_bgr": (0, 0, 255),     "aliases": ["N", "Nasion", "nasion", "L2"]},
    "PNS": {"color_bgr": (255, 0, 0),     "aliases": ["PNS", "pns", "Posterior Nasal Spine"]},
    "Gn":  {"color_bgr": (0, 255, 255),   "aliases": ["Gn", "Gnathion", "gnathion", "L9"]},
    "Po":  {"color_bgr": (0, 255, 0),     "aliases": ["Po", "Porion", "porion", "L4"]},
}
CRITICAL_KEYS = list(CRITICAL_LANDMARKS.keys())

# ---------------------------------------------------------------
# 18 จุดเพิ่มเติม (Po ย้ายไปเป็นจุดหลักแล้ว) — alias สำหรับจับคู่กับ label ใน json
# ---------------------------------------------------------------
ADDITIONAL_LANDMARKS = {
    "Ns":   ["Ns", "N'", "Soft Tissue Nasion", "nasion soft tissue"],
    "Or":   ["Or", "Orbitale", "orbitale", "L3"],
    "ANS":  ["ANS", "Anterior Nasal Spine", "ans"],
    "A":    ["A", "A point", "A-point", "Subspinale", "subspinale", "L5"],
    "Isa":  ["Isa", "U1A", "Upper Incisor Apex", "isa"],
    "Is":   ["Is", "U1", "Upper Incisor", "is"],
    "Ii":   ["Ii", "L1", "Lower Incisor", "ii"],
    "B":    ["B", "B point", "B-point", "Supramentale", "supramental", "L6"],
    "Pg":   ["Pg", "Pogonion", "pogonion", "L7"],
    "Me":   ["Me", "Menton", "menton", "L8"],
    "Iia":  ["Iia", "L1A", "Lower Incisor Apex", "iia"],
    "D":    ["D", "D point", "D-point"],
    "Go":   ["Go", "Gonion", "gonion", "L10"],
    "Ar":   ["Ar", "Articulare", "articulare"],
    "PRN":  ["PRN", "Pronasale", "pronasale", "Prn"],
    "Sn":   ["Sn", "Subnasale", "subnasale"],
    "Ls":   ["Ls", "Labrale Superius", "labrale superius"],
    "Pg'":  ["Pg'", "Pg prime", "Soft Tissue Pogonion", "soft tissue pogonion"],
}

ALL_ALIASES = {**{k: v["aliases"] for k, v in CRITICAL_LANDMARKS.items()}, **ADDITIONAL_LANDMARKS}
ALL_KEYS = list(ALL_ALIASES.keys())   # 23 จุดทั้งหมด

# ---------------------------------------------------------------
# กราฟบาง (sparse) แต่เชื่อมถึงกันหมด (connected) ยึดตามโครงสร้าง
# กายวิภาคจริง — นี่คือกราฟที่ตรงกับหลักการเปเปอร์: จุดที่ "ใกล้" จุดหลัก
# บนกราฟ (hop น้อย) จะได้สีเอนไปทางจุดหลักนั้น ต่างจากจุดที่ "ไกล"
# (Po เป็นจุดหลักที่ 5 อยู่แล้วในกราฟนี้ ผ่าน Or-Po และ Po-Ar)
# ---------------------------------------------------------------
GRAPH_EDGES = [
    ("S", "N"), ("S", "Ar"),
    ("N", "ANS"), ("N", "Or"), ("N", "Ns"),
    ("ANS", "PNS"), ("ANS", "A"),
    ("PNS", "D"),
    ("Or", "Po"),
    ("Po", "Ar"),
    ("Ar", "Go"),
    ("Go", "Me"), ("Go", "D"),
    ("Me", "Gn"), ("Me", "Ii"),
    ("Gn", "Pg"), ("Gn", "Pg'"),
    ("Pg", "B"), ("Pg", "Pg'"),
    ("B", "A"), ("B", "Ii"),
    ("A", "Is"),
    ("Is", "Isa"), ("Is", "Ls"),
    ("Ii", "Iia"),
    ("Ns", "PRN"),
    ("PRN", "Sn"),
    ("Sn", "Ls"),
]


def build_adjacency(edges):
    adj = {}
    for a, b in edges:
        adj.setdefault(a, set()).add(b)
        adj.setdefault(b, set()).add(a)
    return adj


def bfs_distances(start, adj):
    """ระยะทางบนกราฟ (จำนวน hop) จาก start ไปยังทุกจุดที่เชื่อมถึงกัน"""
    dist = {start: 0}
    queue = deque([start])
    while queue:
        cur = queue.popleft()
        for nxt in adj.get(cur, []):
            if nxt not in dist:
                dist[nxt] = dist[cur] + 1
                queue.append(nxt)
    return dist


GRAPH_ADJ = build_adjacency(GRAPH_EDGES)
CRITICAL_DIST = {L: bfs_distances(L, GRAPH_ADJ) for L in CRITICAL_KEYS}  # d(v, L)


def compute_node_colors():
    """
    คำนวณสีของทุกจุดตามสมการ (8)-(9)
    - จุดหลัก (S,N,PNS,Gn): สีคงที่ตามที่กำหนด
    - จุดอื่น: ผสมสีจากจุดหลักตามน้ำหนักถ่วงระยะทางบนกราฟ
    """
    colors = {}
    for key in CRITICAL_KEYS:
        colors[key] = CRITICAL_LANDMARKS[key]["color_bgr"]

    for v in ALL_KEYS:
        if v in CRITICAL_KEYS:
            continue

        weights = {}
        for L in CRITICAL_KEYS:
            d = CRITICAL_DIST[L].get(v)
            if d is None:
                # จุดนี้ไม่เชื่อมถึงจุดหลัก L เลยบนกราฟ -> ตัดออกจากการผสมสี
                continue
            if d == 0:
                d = 1  # กันหารด้วยศูนย์ (ไม่ควรเกิดกับจุดที่ไม่ใช่ critical)
            weights[L] = 1.0 / d

        if not weights:
            # กรณีจุดหลุดออกจากกราฟทั้งหมด (ไม่ควรเกิดถ้า GRAPH_EDGES ครบ) -> ใช้สีเทากลาง
            colors[v] = (128, 128, 128)
            continue

        total_w = sum(weights.values())
        mixed = [0.0, 0.0, 0.0]
        for L, w in weights.items():
            w_hat = w / total_w                      # eq. (8)
            c = CRITICAL_LANDMARKS[L]["color_bgr"]
            for i in range(3):
                mixed[i] += w_hat * c[i]              # eq. (9)
        colors[v] = tuple(int(round(c)) for c in mixed)

    return colors


NODE_COLORS = compute_node_colors()   # คำนวณครั้งเดียว ใช้ร่วมกันทุกภาพ


# ---------------------------------------------------------------
# อ่าน / จับคู่ landmark จาก .json (เหมือน step ก่อนหน้า)
# ---------------------------------------------------------------
def load_landmarks(json_path: str) -> dict:
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    points = {}
    if isinstance(data, dict) and "shapes" in data:
        for shape in data["shapes"]:
            label = shape.get("label")
            pts = shape.get("points")
            if label and pts:
                x, y = pts[0]
                points[label] = (float(x), float(y))
    return points


def match_points(raw_points: dict) -> dict:
    matched = {}
    for key, aliases in ALL_ALIASES.items():
        for alias in aliases:
            if alias in raw_points:
                matched[key] = raw_points[alias]
                break
    return matched


# ---------------------------------------------------------------
# คำนวณมุมทางคลินิก 12 ค่า (SNA, SNB, ANB, SND, NaPg, SN, GoGn,
# NSAr, SArGo, ArGoGn, ArGoN, NGoGn) จากพิกัดใน matched dict
# นิยาม: ANB = SNA-SNB (ตามธรรมเนียมคลินิก) / NaPg = มุม N-A-Pg vertex A
# / SN, GoGn = มุมทิศทางของเส้นเทียบแนวนอนของภาพ
# ---------------------------------------------------------------
ANGLE_NEEDED_KEYS = ["S", "N", "A", "B", "D", "Pg", "Go", "Gn", "Ar"]
ANGLE_COLUMNS = ["SNA", "SNB", "ANB", "SND", "NaPg", "SN", "GoGn",
                  "NSAr", "SArGo", "ArGoGn", "ArGoN", "NGoGn"]


def vertex_angle_deg(vertex, p1, p2):
    v1 = (p1[0] - vertex[0], p1[1] - vertex[1])
    v2 = (p2[0] - vertex[0], p2[1] - vertex[1])
    len1, len2 = math.hypot(*v1), math.hypot(*v2)
    if len1 == 0 or len2 == 0:
        return None
    cos_theta = (v1[0] * v2[0] + v1[1] * v2[1]) / (len1 * len2)
    cos_theta = max(-1.0, min(1.0, cos_theta))
    return math.degrees(math.acos(cos_theta))


def line_orientation_deg(p_from, p_to):
    dx, dy = p_to[0] - p_from[0], p_to[1] - p_from[1]
    return math.degrees(math.atan2(dy, dx))


def compute_angles(matched: dict) -> dict:
    def has(*keys):
        return all(k in matched for k in keys)

    a = {}
    a["SNA"] = vertex_angle_deg(matched["N"], matched["S"], matched["A"]) if has("N", "S", "A") else None
    a["SNB"] = vertex_angle_deg(matched["N"], matched["S"], matched["B"]) if has("N", "S", "B") else None
    a["ANB"] = (a["SNA"] - a["SNB"]) if (a["SNA"] is not None and a["SNB"] is not None) else None
    a["SND"] = vertex_angle_deg(matched["N"], matched["S"], matched["D"]) if has("N", "S", "D") else None
    a["NaPg"] = vertex_angle_deg(matched["A"], matched["N"], matched["Pg"]) if has("A", "N", "Pg") else None
    a["SN"] = line_orientation_deg(matched["S"], matched["N"]) if has("S", "N") else None
    a["GoGn"] = line_orientation_deg(matched["Go"], matched["Gn"]) if has("Go", "Gn") else None
    a["NSAr"] = vertex_angle_deg(matched["S"], matched["N"], matched["Ar"]) if has("S", "N", "Ar") else None
    a["SArGo"] = vertex_angle_deg(matched["Ar"], matched["S"], matched["Go"]) if has("Ar", "S", "Go") else None
    a["ArGoGn"] = vertex_angle_deg(matched["Go"], matched["Ar"], matched["Gn"]) if has("Go", "Ar", "Gn") else None
    a["ArGoN"] = vertex_angle_deg(matched["Go"], matched["Ar"], matched["N"]) if has("Go", "Ar", "N") else None
    a["NGoGn"] = vertex_angle_deg(matched["Go"], matched["N"], matched["Gn"]) if has("Go", "N", "Gn") else None
    return a


# ---------------------------------------------------------------
# วาดเส้น gradient ตามสมการ (10)
# ---------------------------------------------------------------
def draw_gradient_edge(image, p1, c1, p2, c2):
    x1, y1 = p1
    x2, y2 = p2
    D = int(round(math.hypot(x2 - x1, y2 - y1)))

    if D <= 1:
        cv2.line(image, (int(round(x1)), int(round(y1))),
                  (int(round(x2)), int(round(y2))), c1, 1)
        return

    for t in range(D):
        frac = t / (D - 1)
        x = x1 + frac * (x2 - x1)
        y = y1 + frac * (y2 - y1)
        color = tuple(int(round(c1[i] + frac * (c2[i] - c1[i]))) for i in range(3))
        cv2.circle(image, (int(round(x)), int(round(y))), EDGE_DOT_RADIUS, color, -1)


# ---------------------------------------------------------------
# ประมวลผล 1 ภาพ: วาดกราฟ (edges + nodes) ด้วยสีตาม AIT + copy json
# ---------------------------------------------------------------
def process_one_sample(png_path: str, out_dir: str):
    stem = Path(png_path).stem
    json_path = str(Path(png_path).with_suffix(".json"))

    image = cv2.imread(png_path)
    if image is None:
        print(f"⚠️ อ่านภาพไม่ได้: {png_path}")
        return

    if not os.path.exists(json_path):
        print(f"⚠️ ไม่พบไฟล์ json คู่กัน: {json_path}")
        return

    raw_points = load_landmarks(json_path)
    matched = match_points(raw_points)

    missing = [k for k in ALL_KEYS if k not in matched]
    if missing:
        print(f"⚠️ [{stem}] หาจุดไม่เจอ ({len(missing)}): {missing}")

    out_img = image.copy()

    # 1) วาดเส้น edge แบบ gradient — วาดก่อน เพื่อให้ node ทับอยู่ด้านบนสุด
    for a, b in GRAPH_EDGES:
        if a in matched and b in matched:
            p1, p2 = matched[a], matched[b]
            c1, c2 = NODE_COLORS[a], NODE_COLORS[b]
            draw_gradient_edge(out_img, p1, c1, p2, c2)

    # 2) วาดจุด (node) ด้วยสีที่คำนวณจาก AIT
    for key, (x, y) in matched.items():
        color = NODE_COLORS[key]
        center = (int(round(x)), int(round(y)))
        cv2.circle(out_img, center, NODE_RADIUS, color, thickness=-1)

    os.makedirs(out_dir, exist_ok=True)
    out_png_path = os.path.join(out_dir, f"{stem}.png")      # ชื่อไฟล์เดิม เหมือน step ก่อนหน้า
    ok = cv2.imwrite(out_png_path, out_img)
    if not ok:
        raise IOError(f"cv2.imwrite เขียนไฟล์ไม่สำเร็จ: {out_png_path}")

    out_json_path = os.path.join(out_dir, f"{stem}.json")
    shutil.copy(json_path, out_json_path)


# ---------------------------------------------------------------
# วนทุก Fold x ทุก split x ทุกภาพ (รันขนานด้วย multiprocessing)
# ---------------------------------------------------------------
def process_all_folds_parallel(max_workers: int = 8):
    fold_dirs = sorted(glob.glob(os.path.join(SPLIT_ROOT, "Fold*")))
    if not fold_dirs:
        print(f"⚠️ ไม่พบโฟลเดอร์ Fold ใน {SPLIT_ROOT}")
        return

    tasks = []
    for fold_dir in fold_dirs:
        fold_name = os.path.basename(fold_dir)
        for split_name in ["Train", "Val", "Test"]:
            split_dir = os.path.join(fold_dir, split_name)
            png_files = sorted(glob.glob(os.path.join(split_dir, "*.png")))
            if not png_files:
                continue
            out_dir = os.path.join(PREVIEW_DIR, fold_name, split_name)
            os.makedirs(out_dir, exist_ok=True)
            tasks.extend([(p, out_dir) for p in png_files])

    print(f"🚀 เริ่มประมวลผล {len(tasks)} ภาพ ด้วย {max_workers} processes")

    success_count = 0
    fail_count = 0
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(process_one_sample, png_path, out_dir): png_path
                   for png_path, out_dir in tasks}
        for future in as_completed(futures):
            png_path = futures[future]
            try:
                future.result()   # ถ้ามี exception ใน worker จะถูกโยนขึ้นมาที่นี่
                success_count += 1
            except Exception as e:
                fail_count += 1
                print(f"❌ ล้มเหลว: {png_path} -> {type(e).__name__}: {e}")

    print(f"✅ เสร็จสิ้น สำเร็จ {success_count} ภาพ / ล้มเหลว {fail_count} ภาพ → {PREVIEW_DIR}")


# ---------------------------------------------------------------
# คำนวณมุมทั้ง 12 ค่า จาก Fold1 เท่านั้น (Train+Val+Test) -> export Excel
# ---------------------------------------------------------------
def collect_angles_fold1_and_export():
    rows = []
    for split_name in ["Train", "Val", "Test"]:
        split_dir = os.path.join(SPLIT_ROOT, ANGLE_SOURCE_FOLD, split_name)
        json_files = sorted(glob.glob(os.path.join(split_dir, "*.json")))

        for json_path in json_files:
            stem = Path(json_path).stem
            raw_points = load_landmarks(json_path)
            matched = match_points(raw_points)

            missing = [k for k in ANGLE_NEEDED_KEYS if k not in matched]
            angles = compute_angles(matched)

            row = {"image_id": stem, "split": split_name}
            row.update(angles)
            row["missing_landmarks"] = ", ".join(missing) if missing else ""
            rows.append(row)

            if missing:
                print(f"⚠️ [มุม][{stem}] หาจุดไม่เจอ: {missing}")

    df = pd.DataFrame(rows, columns=["image_id", "split"] + ANGLE_COLUMNS + ["missing_landmarks"])

    os.makedirs(os.path.dirname(ANGLE_XLSX), exist_ok=True)
    df.to_excel(ANGLE_XLSX, index=False, sheet_name="Angles", engine="openpyxl")

    wb = load_workbook(ANGLE_XLSX)
    ws = wb["Angles"]
    header_font = Font(name="Arial", bold=True)
    body_font = Font(name="Arial")

    for col_idx, col_name in enumerate(df.columns, start=1):
        cell = ws.cell(row=1, column=col_idx)
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center")
        max_len = max(len(str(col_name)), df[col_name].astype(str).map(len).max() if len(df) else 0)
        ws.column_dimensions[get_column_letter(col_idx)].width = min(max(max_len + 2, 10), 40)

    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.font = body_font

    ws.freeze_panes = "A2"
    wb.save(ANGLE_XLSX)

    if not os.path.exists(ANGLE_XLSX):
        raise IOError(f"บันทึกไฟล์ไม่สำเร็จ ไม่พบไฟล์ที่ {ANGLE_XLSX}")

    file_size = os.path.getsize(ANGLE_XLSX)
    print(f"✅ คำนวณมุมเสร็จสิ้น รวม {len(df)} ภาพ ({ANGLE_SOURCE_FOLD}) → {ANGLE_XLSX} ({file_size:,} bytes)")


if __name__ == "__main__":
    process_all_folds_parallel(max_workers=8)   # ภาพ: ทำทุก Fold
    collect_angles_fold1_and_export()           # มุม: ทำเฉพาะ Fold1 -> Excel

🚀 เริ่มประมวลผล 2000 ภาพ ด้วย 8 processes


✅ เสร็จสิ้น สำเร็จ 2000 ภาพ / ล้มเหลว 0 ภาพ → /home/dl-box/users/students/phasathorn-jewrasumnuay/CephPic/A5AITcolorFrom4color
✅ คำนวณมุมเสร็จสิ้น รวม 400 ภาพ (Fold1) → /home/dl-box/users/students/phasathorn-jewrasumnuay/CephPic/A5_CephalometricAngles.xlsx (76,844 bytes)
